In [1]:
import numpy as np

# Note: Following standard literature (e.g., Cary et al., 2007; Edelman et al., 2007),
# we assume equal ad quality scores (rho_i = 1) without loss of generality.

def rank(b, rho=None):
    """Sort bidders by ranking score w = rho*b (ties resolved by index)."""
    if rho is None:
        rho = np.ones(len(b))
    w = rho * b
    order = np.argsort(-w, kind="stable")
    return order, w[order]

def slot_of(b, p, rho=None):
    """Returns slot index (0-indexed) held by each agent, or -1 if no slot."""
    if rho is None:
        rho = np.ones(len(b))
    order, _ = rank(b, rho)
    m = min(len(b), len(p))
    slot = np.full(len(b), -1)
    slot[order[:m]] = np.arange(m)
    return slot

def welfare(w, p):
    """Maximum reported social welfare."""
    w_sorted = np.sort(w)[::-1]
    m = min(len(w), len(p))
    return w_sorted[:m] @ p[:m]

def gsp_price(b, p, rho=None):
    """Per-click GSP price paid by each bidder."""
    if rho is None:
        rho = np.ones(len(b))
    order, w = rank(b, rho)
    w_next = np.append(w, 0)[1:]
    price = np.zeros(len(b))
    for j in range(min(len(b), len(p))):
        price[order[j]] = w_next[j] / rho[order[j]]
    return price

def vcg_payment(b, p, rho=None):
    """Total expected VCG payment per impression."""
    if rho is None:
        rho = np.ones(len(b))
    w, slot = rho * b, slot_of(b, p, rho)
    pay = np.zeros(len(b))
    for i in range(len(b)):
        own = p[slot[i]] * w[i] if slot[i] >= 0 else 0
        pay[i] = welfare(np.delete(w, i), p) - (welfare(w, p) - own)
    return pay

def vcg_price(b, p, rho=None):
    """Per-click VCG price = payment / p_slot."""
    if rho is None:
        rho = np.ones(len(b))
    slot = slot_of(b, p, rho)
    pay = vcg_payment(b, p, rho)
    price = np.zeros(len(b))
    for i in np.flatnonzero(slot >= 0):
        price[i] = pay[i] / (p[slot[i]] * rho[i])
    return price

def utility(v, b, p, price_rule, rho=None):
    """Per-impression utility: p_slot * (v_i - price_i)."""
    if rho is None:
        rho = np.ones(len(b))
    slot = slot_of(b, p, rho)
    price = price_rule(b, p, rho)
    u = np.zeros(len(b))
    for i in np.flatnonzero(slot >= 0):
        u[i] = rho[i] * p[slot[i]] * (v[i] - price[i])
    return u

def bidder_optimal_bids(v, p):
    """Bidder-Optimal (Lowest Revenue / VCG Equivalent) LEFE GSP Equilibrium Bids."""
    n = len(v)
    order = np.argsort(-v, kind="stable")
    v_sorted = v[order]
    m = min(n, len(p))
    beta = v_sorted.copy()
    for j in range(m - 1, 0, -1):
        r = p[j] / p[j - 1]
        beta_next = beta[j + 1] if j + 1 < n else 0.0
        beta[j] = (1 - r) * v_sorted[j] + r * beta_next
    b = np.zeros(n)
    b[order] = beta
    return b

def seller_optimal_bids(v, p):
    """Seller-Optimal (Highest Revenue) LEFE GSP Equilibrium Bids (Edelman et al., 2007)."""
    n = len(v)
    order = np.argsort(-v, kind="stable")
    v_sorted = v[order]
    m = min(n, len(p))
    
    p_price = np.zeros(m)
    p_next = v_sorted[m] if m < n else 0.0
    
    for j in range(m - 1, -1, -1):
        p_ratio = p[j + 1] / p[j] if j + 1 < len(p) else 0.0
        p_price[j] = p_next * p_ratio + v_sorted[j] * (1.0 - p_ratio)
        p_next = p_price[j]
        
    b = np.zeros(n)
    b[order[0]] = v_sorted[0]
    for j in range(1, m):
        b[order[j]] = p_price[j - 1]
    for j in range(m, n):
        b[order[j]] = v_sorted[j]
    return b

In [2]:
def verify_nash_equilibrium(v, p, b_target, grid_step=0.01, tol=1e-6):
    """Verifies zero unilateral deviation gains under GSP at b_target (Grid fix included)."""
    n = len(v)
    max_val = np.max(v) * 1.5
    grid = np.arange(0, max_val + grid_step, grid_step)
    
    is_nash = True
    max_dev_gains = []
    
    for i in range(n):
        u_curr = utility(v, b_target, p, gsp_price)[i]
        b_candidate = b_target.copy()
        grid_i = np.unique(np.append(grid, v[i]))  # Explicitly include true v_i
        
        gains = []
        for x in grid_i:
            b_candidate[i] = x
            u_alt = utility(v, b_candidate, p, gsp_price)[i]
            gains.append(u_alt - u_curr)
            
        max_gain = np.max(gains)
        max_dev_gains.append(max_gain)
        if max_gain > tol:
            is_nash = False
            
    return is_nash, max_dev_gains

def verify_vcg_dsic(v, p, num_samples=100, grid_step=0.01, tol=1e-8):
    """Verifies VCG DSIC for bidder i against N=100 arbitrary opposing bid vectors."""
    n = len(v)
    rng = np.random.default_rng(42)
    max_val = np.max(v) * 2.0
    base_grid = np.arange(0, max_val + grid_step, grid_step)
    
    violations = 0
    max_gain = 0.0
    
    for i in range(n):
        grid_i = np.unique(np.append(base_grid, v[i]))  # Explicitly include true v_i
        for _ in range(num_samples):
            b_opp = rng.uniform(0, max_val, n)
            
            b_truth = b_opp.copy()
            b_truth[i] = v[i]
            u_truth = utility(v, b_truth, p, vcg_price)[i]
            
            for x in grid_i:
                b_cand = b_opp.copy()
                b_cand[i] = x
                u_cand = utility(v, b_cand, p, vcg_price)[i]
                gain = u_cand - u_truth
                if gain > tol:
                    violations += 1
                    max_gain = max(max_gain, gain)
                    
    return violations, max_gain

# Execution Check
v_demo = np.array([10.0, 8.0, 6.0, 4.0, 2.0])
p_demo = np.array([0.30, 0.20, 0.10])
b_opt = bidder_optimal_bids(v_demo, p_demo)

is_nash, dev_gains = verify_nash_equilibrium(v_demo, p_demo, b_opt)
vcg_violations, max_dsic_gain = verify_vcg_dsic(v_demo, p_demo)

print(f"Bidder-Optimal GSP Nash Equilibrium Verified? {is_nash} (Max Gain: {max(dev_gains):.2e})")
print(f"VCG DSIC Verified across 100 Random Opposing Environments? {vcg_violations == 0} (Violations: {vcg_violations})")

Bidder-Optimal GSP Nash Equilibrium Verified? True (Max Gain: 0.00e+00)
VCG DSIC Verified across 100 Random Opposing Environments? True (Violations: 0)


In [3]:
DIVERGED_FACTOR = 1e3

def slot_prices(b, j, K):
    others = np.sort(np.delete(b, j))[::-1]
    return np.array([others[s] if s < len(others) else 0.0 for s in range(K)])

def current_slot(b, j):
    return sum(b[i] > b[j] or (b[i] == b[j] and i < j) for i in range(len(b)))

def balanced_bid(j, b, v, theta, restricted):
    K = len(theta)
    price = slot_prices(b, j, K)
    utility_vals = theta * (v[j] - price)
    cur = current_slot(b, j)
    first = cur if (restricted and cur < K) else 0
    s = first + int(np.argmax(utility_vals[first:]))
    if utility_vals[s] <= 0:
        return v[j]
    theta_above = 2 * theta[0] if s == 0 else theta[s - 1]
    return v[j] - theta[s] / theta_above * (v[j] - price[s])

def next_bids(b, v, theta, restricted):
    return np.array([balanced_bid(j, b, v, theta, restricted) for j in range(len(b))])

def is_fixed_point(b, v, theta, restricted, tol=1e-9):
    return np.max(np.abs(next_bids(b, v, theta, restricted) - b)) <= tol

def diverged(b, v):
    return (not np.all(np.isfinite(b))) or b.max() > DIVERGED_FACTOR * max(v)

def run_synchronous(v, theta, b0, restricted, max_rounds=10000, tol=1e-9):
    b = np.array(b0, dtype=float)
    history = [b.copy()]
    seen = {tuple(np.round(b, 9)): 0}
    
    for t in range(1, max_rounds + 1):
        new_b = next_bids(b, v, theta, restricted)
        if np.max(np.abs(new_b - b)) <= tol:
            return {"outcome": "converged", "steps": t, "epochs": float(t), "bids": new_b}
        b = new_b
        history.append(b.copy())
        if diverged(b, v):
            return {"outcome": "diverged", "steps": t, "epochs": float(t), "bids": b}
        key = tuple(np.round(b, 9))
        if key in seen:
            prev_t = seen[key]
            period = t - prev_t
            cycle_orbit = np.array(history[prev_t:t])
            return {
                "outcome": "cycle", 
                "steps": t, 
                "epochs": float(t), 
                "bids": b, 
                "period": period, 
                "cycle_orbit": cycle_orbit
            }
        seen[key] = t
    return {"outcome": "undecided", "steps": max_rounds, "epochs": float(max_rounds), "bids": b}

def run_random_asynchronous(v, theta, b0, restricted, rng, max_steps=30000, tol=1e-9, window=2000):
    n = len(v)
    b = np.array(b0, dtype=float)
    tail = []
    for t in range(1, max_steps + 1):
        j = int(rng.integers(n))
        new = balanced_bid(j, b, v, theta, restricted)
        moved = abs(new - b[j]) > tol
        b[j] = new
        if diverged(b, v):
            return {"outcome": "diverged", "steps": t, "epochs": t / n, "bids": b.copy()}
        if not moved and is_fixed_point(b, v, theta, restricted, tol):
            return {"outcome": "converged", "steps": t, "epochs": t / n, "bids": b.copy()}
        if t > max_steps - window:
            tail.append(b.copy())
    swing = np.ptp(np.array(tail), axis=0).max()
    return {"outcome": "cycle" if swing > 1e-6 else "undecided", "steps": max_steps, "epochs": max_steps / n, "bids": b.copy()}

In [4]:
from collections import defaultdict

def run_experiment(num_trials=200, n=5, K=3, seed=0, init_type="random"):
    baseline_stats = defaultdict(list)
    outcomes_count = defaultdict(lambda: defaultdict(int))
    convergence_epochs = defaultdict(list)
    eq_distances = defaultdict(list)
    
    converged_econ = defaultdict(list)
    cycling_econ = defaultdict(list)

    for trial in range(num_trials):
        rng = np.random.default_rng(seed + trial)
        v = rng.uniform(1, 10, n)
        theta = np.sort(rng.uniform(0.05, 1, K))[::-1]
        
        if init_type == "random":
            b0 = rng.uniform(0, 1, n) * v
        elif init_type == "truthful":
            b0 = v.copy()
        elif init_type == "zero":
            b0 = np.zeros(n)

        max_welfare = welfare(v, theta)
        b_target = bidder_optimal_bids(v, theta)
        p_target = gsp_price(b_target, theta)
        
        # Fixed: eval_bids now accepts price_fn
        def eval_bids(b_vec, price_fn=gsp_price):
            slots = slot_of(b_vec, theta)
            prices = price_fn(b_vec, theta)
            rev, surplus, realized_welfare = 0.0, 0.0, 0.0
            for i in range(n):
                s = slots[i]
                if 0 <= s < len(theta):
                    p_val = theta[s]
                    rev += prices[i] * p_val
                    surplus += p_val * (v[i] - prices[i])
                    realized_welfare += p_val * v[i]
            return {
                "revenue": rev, 
                "surplus": surplus, 
                "efficiency": realized_welfare / max_welfare if max_welfare > 0 else 0.0
            }

        # Static Baselines (Fixed VCG pricing call)
        baselines = {
            "VCG": eval_bids(v, price_fn=vcg_price),
            "GSP Bidder-Opt": eval_bids(b_target, price_fn=gsp_price),
            "GSP Seller-Opt": eval_bids(seller_optimal_bids(v, theta), price_fn=gsp_price),
            "GSP Truthful": eval_bids(v, price_fn=gsp_price)
        }
        for mech, m_data in baselines.items():
            for met_name, val in m_data.items():
                baseline_stats[(mech, met_name)].append(val)

        # Dynamic Runs
        for alg, restricted in (("BB", False), ("RBB", True)):
            for sched, run_func in (("synchronous", lambda: run_synchronous(v, theta, b0, restricted)),
                                      ("random async", lambda: run_random_asynchronous(v, theta, b0, restricted, rng))):
                res = run_func()
                key = (alg, sched)
                outcomes_count[key][res["outcome"]] += 1
                
                if res["outcome"] == "converged":
                    convergence_epochs[key].append(res["epochs"])
                    
                    # Fixed: Distance measured on actual GSP prices paid (P_final vs P_target)
                    p_final = gsp_price(res["bids"], theta)
                    dist = np.max(np.abs(p_final - p_target))
                    eq_distances[key].append(dist)
                    
                    m_eval = eval_bids(res["bids"])
                    for met_name, val in m_eval.items():
                        converged_econ[(key, met_name)].append(val)
                        
                elif res["outcome"] == "cycle":
                    if "cycle_orbit" in res:
                        orbit_evals = [eval_bids(b_state) for b_state in res["cycle_orbit"]]
                        avg_rev = np.mean([e["revenue"] for e in orbit_evals])
                        avg_surp = np.mean([e["surplus"] for e in orbit_evals])
                        avg_eff = np.mean([e["efficiency"] for e in orbit_evals])
                    else:
                        e = eval_bids(res["bids"])
                        avg_rev, avg_surp, avg_eff = e["revenue"], e["surplus"], e["efficiency"]
                        
                    cycling_econ[(key, "revenue")].append(avg_rev)
                    cycling_econ[(key, "surplus")].append(avg_surp)
                    cycling_econ[(key, "efficiency")].append(avg_eff)

    print(f"\n==========================================================================================")
    print(f" EXPERIMENT RESULTS: {num_trials} Trials (n={n}, K={K}, Init={init_type})")
    print(f"==========================================================================================")
    
    print("\n1. DYNAMIC CONVERGENCE & EQUILIBRIUM ACCURACY")
    print(f"{'Alg / Sched':<22}{'Conv':<6}{'Cycle':<7}{'Med Epochs':<12}{'Mean Epochs':<12}{'Price Dist to P*':<18}{'Exact P* Conv %':<15}")
    print("-" * 92)
    for key in (("BB", "synchronous"), ("BB", "random async"), ("RBB", "synchronous"), ("RBB", "random async")):
        alg, sched = key
        c = outcomes_count[key]
        ep = convergence_epochs[key]
        dist = eq_distances[key]
        
        med_ep = f"{np.median(ep):.1f}" if ep else "-"
        mean_ep = f"{np.mean(ep):.1f}" if ep else "-"
        mean_d = f"{np.mean(dist):.4f}" if dist else "-"
        exact_pct = f"{np.mean(np.array(dist) < 1e-3) * 100:.1f}%" if dist else "-"
        
        label = f"{alg} ({sched})"
        print(f"{label:<22}{c['converged']:<6}{c['cycle']:<7}{med_ep:<12}{mean_ep:<12}{mean_d:<18}{exact_pct:<15}")

    print("\n2. STATIC BASELINE BOUNDS")
    print(f"{'Baseline Mechanism':<22}{'Avg Revenue':<15}{'Avg Surplus':<15}{'Avg Efficiency':<15}")
    print("-" * 67)
    for mech in ("VCG", "GSP Bidder-Opt", "GSP Seller-Opt", "GSP Truthful"):
        r = np.mean(baseline_stats[(mech, "revenue")])
        s = np.mean(baseline_stats[(mech, "surplus")])
        e = np.mean(baseline_stats[(mech, "efficiency")])
        print(f"{mech:<22}{r:<15.2f}{s:<15.2f}{e:<15.4f}")

    print("\n3. CONVERGED RUNS: ECONOMIC METRICS")
    print(f"{'Alg / Sched':<22}{'Avg Revenue':<15}{'Avg Surplus':<15}{'Avg Efficiency':<15}")
    print("-" * 67)
    for key in (("BB", "synchronous"), ("BB", "random async"), ("RBB", "synchronous"), ("RBB", "random async")):
        alg, sched = key
        label = f"{alg} ({sched})"
        if converged_econ[(key, "revenue")]:
            r = np.mean(converged_econ[(key, "revenue")])
            s = np.mean(converged_econ[(key, "surplus")])
            e = np.mean(converged_econ[(key, "efficiency")])
            print(f"{label:<22}{r:<15.2f}{s:<15.2f}{e:<15.4f}")
        else:
            print(f"{label:<22}{'No Converged Runs':<45}")

In [5]:
# --- Sweep 1: Initial Bids Sensitivity ---
print("\n================ RUNNING INITIALIZATION SWEEP ================")
for init_mode in ["random", "truthful", "zero"]:
    run_experiment(num_trials=100, n=5, K=3, seed=0, init_type=init_mode)

# --- Sweep 2: Market Size & Competition Grid ---
print("\n================ RUNNING PARAMETER GRID SWEEP ================")
parameter_grid = [
    (3, 2),  # Small market, surplus bidders
    (5, 3),  # Medium market
    (5, 5),  # Equal slots and bidders (n = K)
    (10, 5)  # Large market, high competition
]

for n_val, K_val in parameter_grid:
    run_experiment(num_trials=100, n=n_val, K=K_val, seed=0, init_type="random")


================ RUNNING INITIALIZATION SWEEP ================

 EXPERIMENT RESULTS: 100 Trials (n=5, K=3, Init=random)

1. DYNAMIC CONVERGENCE & EQUILIBRIUM ACCURACY
Alg / Sched           Conv  Cycle  Med Epochs  Mean Epochs Price Dist to P*  Exact P* Conv %
--------------------------------------------------------------------------------------------
BB (synchronous)      97    3      10.0        24.9        0.0000            100.0%         
BB (random async)     100   0      10.1        23.3        0.0000            100.0%         
RBB (synchronous)     100   0      11.0        26.8        0.0000            100.0%         
RBB (random async)    100   0      7.6         15.3        0.0000            100.0%         

2. STATIC BASELINE BOUNDS
Baseline Mechanism    Avg Revenue    Avg Surplus    Avg Efficiency 
-------------------------------------------------------------------
VCG                   8.07           4.00           1.0000         
GSP Bidder-Opt        8.07           4.00  